# Mimicking Finance — single-panel replication

Runs the LSTM trade-direction replication on **one holdings parquet**. Toggle **per-fund** vs **global** LSTM.
Tuned for **CPU (~20 cores, limited RAM)**: per-fund training is parallelised across cores and fully mini-batched.
All logic lives in `pipeline.py`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import pipeline as P

## 1. Configure (CPU, memory-safe)
`eval_mode='predictive'` 用 `future_2q_ret`：accuracy(t) 在 t+1 揭晓，收益取 t+1→t+2 → **不重叠**。
`parallel_backend='threading'` 共享内存避免 OOM。`n_slots=25` 匹配这个文件。

In [ ]:
cfg = P.Config(
    data_path='manager_holdings/master_batches_returnfiltered/panel_holdings_All_Funds_filter_rank.parquet',
    inv_type_codes=(401,),         # None=全部；给多个(如 (401,402))才能启用「类别内 peer」
    max_rank=None,                 # None=用文件里现有的；设 10 只用前10大持仓
    eval_mode='predictive',        # 'tradeable' 最严（t+2->t+3，含披露延迟）
    model_mode='per_fund',
    device='cpu', parallel_backend='threading', n_jobs=-1, batch=4096,
    downcast=True, keep_panel=False,
    min_years=7, min_holdings=10, change_band=0.01,
    window_q=28, train_q=20, test_q=8, seq_len=8, step=8,
    max_epochs=25, hidden_cap=64, lr=3e-3,
    out_dir='outputs',
)
cfg

## 2. Run the full pipeline
(data prep → sequences → LSTM walk-forward → evaluation → figures)

In [ ]:
res = P.run(cfg)

## 3. Predictability (vs naive baseline)  — paper: LSTM 0.71, naive 0.52

In [ ]:
m = res['metrics']
pd.Series({k: round(m[k],4) for k in ['lstm_precision_pooled','naive_precision_pooled',
    'lstm_precision_fundavg','naive_precision_fundavg','n_funds','n_predictions']}).to_frame('value')

## 4. Table X — fund quintiles on predictability
Benchmark-adjusted cumulative abnormal return, 1–4 quarters. Q1=least predictable, Q5=most. Paper Q5−Q1 = −0.79% (t=−3.05).

In [ ]:
res['tables']['tableX'].round(4)

## 5. Table XI — correct vs incorrect positions  (paper diff −0.23%/qtr, t=−12.4)

In [ ]:
res['tables']['tableXI'].round(4)

## 6. Table XII — stock quintiles on cross-fund accuracy  (paper Q1−Q5 = +1.06%/qtr, t=5.74)

In [ ]:
res['tables']['tableXII'].round(4)

## 7. Figures

In [ ]:
res['figures']['precision_dist']

In [ ]:
res['figures']['stock_ls']

## 8. Compare per-fund vs global
Per-fund usually lifts LSTM precision **above** naive; global tends to **tie** it.

In [ ]:
cfg_g = P.Config(**{**{k:v for k,v in res['config'].items() if k!='col_map'},
                    'model_mode':'global','out_dir':'outputs_global'})
cfg_g.col_map = P.Config().col_map
res_g = P.run(cfg_g)
pd.DataFrame({
    'per_fund':[res['metrics']['lstm_precision_fundavg'], res['metrics']['naive_precision_fundavg'], res['metrics']['tableXII_Q1mQ5']],
    'global':  [res_g['metrics']['lstm_precision_fundavg'], res_g['metrics']['naive_precision_fundavg'], res_g['metrics']['tableXII_Q1mQ5']]},
    index=['LSTM prec (fund-avg)','Naive prec (fund-avg)','Table XII Q1-Q5']).round(4)

## 9. 口径稳健性检验（关键）
训练与评估口径无关 → 下面四种口径**共用同一次训练**。

| eval_mode | 排序 | 收益窗口 | 含义 |
|---|---|---|---|
| `contemporaneous` | acc(t) | t→t+1 | **重叠** → 同期共动（有偏，仅对照）|
| `lagged` | acc(t−1) | t→t+1 | 干净但信号旧 |
| `predictive` | acc(t) | t+1→t+2 | 干净且最新 ← 默认 |
| `tradeable` | acc(t) | t+2→t+3 | **最严**：额外跨过 45–60 天披露延迟 |

**如果价差只在 `contemporaneous` 显著、换成 `predictive`/`tradeable` 就消失** → 论文的经济学结论是同期相关，不是预测能力。

In [ ]:
P.compare_eval_modes(res['predictions'], cfg).round(4)

## 10. 不同 rank 维度的对比
`max_rank` 改变的是**面板本身**（哪些持仓存在），所以每个 cutoff 必须各自重训一次（不像 `eval_mode` 是免费的）。

读法：cutoff 越小 → 只留经理最大、最有信心的持仓。论文认为大仓位交易更主动、**更难预测** → 预测准确率应该随 cutoff 变小而下降。

注意：`min_holdings` 会自动被下调到不超过 `max_rank`（否则 `max_rank=10` + `min_holdings=10` 会把面板清空）。

In [ ]:
summary, results = P.run_rank_sweep(cfg, ranks=(10, 25))
summary.round(4)

## 11. 按 InvTypeCode 分组对比
`inv_type_codes` 既是**样本筛选**，也决定了 peer 特征和 Table X 的 benchmark 口径：

- 给**单个** code（如 `(401,)`）→ 全样本同类，peer rate 退化成全市场（无横截面变异），benchmark = 全样本均值
- 给**多个** code → peer rate 在**类别内**计算、benchmark 用**同类基金**（贴近论文的 Morningstar category 口径）

In [ ]:
from dataclasses import replace
import pandas as pd
rows=[]
for codes in [(401,), (402,), None]:      # None = 全部类别一起（启用类别内 peer）
    r = P.run(replace(cfg, inv_type_codes=codes, out_dir=f'outputs_itc_{codes}'), verbose=False)
    m = r['metrics']
    rows.append({'inv_type_codes': str(codes), 'n_funds': m['n_funds'],
                 'LSTM_prec': m['lstm_precision_fundavg'], 'naive_prec': m['naive_precision_fundavg'],
                 'benchmark': m['benchmark'],
                 'XII_Q1mQ5': m['tableXII_Q1mQ5'], 'XII_t': m['tableXII_Q1mQ5_t']})
pd.DataFrame(rows).round(4)